## BARAM 2026 - Serial Residual + Flow Ablation

원우의 직렬 잔차 구조에서 아이디어만 참고하고, 우리 데이터·외부 예보·Flow 피처로 독립 구현한 실험입니다.

- 1단계: 풍속 중심 MTMLP가 capacity factor base를 예측
- 2단계: LightGBM이 `target CF - DL base CF` 잔차를 학습
- 통제 실험: 잔차 LightGBM의 입력만 `C0` 대 `C0 + Flow 12개`로 변경
- DL 구조·시드·후처리·검증 fold는 두 실험에서 고정

### 1. Colab·Drive 설정

In [ ]:
import os, sys, json, hashlib, random, re, time, pickle, platform, subprocess
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "lightgbm", "xgboost",
         "torch", "tqdm", "pyarrow", "openpyxl", "eccodes"],
        check=True,
    )

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgbm
from tqdm.auto import tqdm

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/BARAM2026/data")
    BASE_RUN_ROOT = Path("/content/drive/MyDrive/BARAM2026/hyejin_external_champion")
    RUN_ROOT = Path("/content/drive/MyDrive/BARAM2026/hyejin_serial_residual_flow")
else:
    DATA_DIR = Path.cwd() / "data"
    BASE_RUN_ROOT = Path.cwd() / "runs" / "hyejin_external_champion"
    RUN_ROOT = Path.cwd() / "runs" / "hyejin_serial_residual_flow"

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
EXTERNAL_DIR = DATA_DIR / "external"
BASE_CACHE_DIR = BASE_RUN_ROOT / "cache"
CACHE_DIR = RUN_ROOT / "cache"
for path in [RUN_ROOT, EXTERNAL_DIR, BASE_CACHE_DIR, CACHE_DIR]:
    path.mkdir(exist_ok=True, parents=True)

# Open-Meteo 무료 API의 비상업 이용 조건을 팀에서 확인한 뒤 True로 바꿉니다.
ACK_OPEN_METEO_TERMS = False  #@param {type:"boolean"}
FORCE_JMA_FETCH = False
FORCE_ECMWF_FETCH = False
FINAL_GBM_SEEDS = 3
FINAL_DL_SEEDS = 3
FLOW_VERSION = "v1"
RUN_FINAL_IF_PASS = True
PASS_CV_DELTA = 0.0005
PASS_MIN_POSITIVE_FOLDS = 3
PASS_MIN_SUBMETRIC_DELTA = -0.0005

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("device:", DEVICE, "/", GPU_NAME)
if IN_COLAB and "A100" not in GPU_NAME.upper():
    print("[주의] A100 런타임인지 확인하세요.")

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {"kpx_group_1": 21600, "kpx_group_2": 21600, "kpx_group_3": 21000}
GROUP_SCALES = {1: 0.98, 2: 0.98, 3: 0.97}

REQUIRED = [
    "train/train_labels.csv", "train/ldaps_train.csv", "train/gfs_train.csv",
    "train/scada_vestas_train.csv", "train/scada_unison_train.csv",
    "test/ldaps_test.csv", "test/gfs_test.csv", "info.xlsx", "sample_submission.csv",
]
missing = [rel for rel in REQUIRED if not (DATA_DIR / rel).exists()]
assert not missing, "누락된 공식 파일:\n" + "\n".join(missing)

def sha256_file(path, block=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(block)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

print("data    :", DATA_DIR)
print("external:", EXTERNAL_DIR)
print("base cache:", BASE_CACHE_DIR)
print("output    :", RUN_ROOT)

### 2. 기본 데이터셋 생성 함수

In [ ]:
# 1. v10 데이터셋
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def dms_to_decimal(s):
    pattern = r"(\d+)°(\d+)'([\d.]+)\"([NS])\s+(\d+)°(\d+)'([\d.]+)\"([EW])"
    m = re.match(pattern, s.strip())
    d1, m1, s1, hemi1, d2, m2, s2, hemi2 = m.groups()
    lat = float(d1) + float(m1) / 60 + float(s1) / 3600
    if hemi1 == "S": lat = -lat
    lon = float(d2) + float(m2) / 60 + float(s2) / 3600
    if hemi2 == "W": lon = -lon
    return lat, lon

def find_nearest_grid(weather_df, coord_df, name_col, lat_col="lat", lon_col="lon"):
    grids = weather_df[["grid_id", "latitude", "longitude"]].drop_duplicates()
    mapping = {}
    for _, row in coord_df.iterrows():
        dists = haversine(row[lat_col], row[lon_col], grids["latitude"], grids["longitude"])
        mapping[row[name_col]] = grids.loc[dists.idxmin(), "grid_id"]
    return mapping

def add_wind_features(df, uv_pairs):
    df = df.copy()
    for u_col, v_col, height in uv_pairs:
        df[f"ws_{height}"] = np.sqrt(df[u_col] ** 2 + df[v_col] ** 2)
        wd_rad = np.arctan2(df[v_col], df[u_col])
        df[f"wd_sin_{height}"] = np.sin(wd_rad)
        df[f"wd_cos_{height}"] = np.cos(wd_rad)
    return df

def build_group_weather(weather_df, group_grid_map, prefix):
    weather_df = weather_df.copy()
    weather_df["forecast_kst_dtm"] = pd.to_datetime(weather_df["forecast_kst_dtm"])
    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in weather_df.columns if c not in {"forecast_kst_dtm", *drop_cols}]
    frames = []
    for group, grid_id in group_grid_map.items():
        sub = weather_df.loc[weather_df["grid_id"] == grid_id, ["forecast_kst_dtm", *value_cols]].copy()
        sub = sub.rename(columns={c: f"{prefix}_{c}" for c in value_cols})
        sub["kpx_group"] = int(group)
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

def add_forecast_disagreement(df):
    df = df.copy()
    df["ws_disagreement"] = np.abs(df["ldaps_ws_10m"] - df["gfs_ws_10m"])
    df["wd_agreement"] = df["ldaps_wd_cos_10m"] * df["gfs_wd_cos_10m"] + df["ldaps_wd_sin_10m"] * df["gfs_wd_sin_10m"]
    return df

def add_seasonal_interaction(df):
    df = df.copy()
    month = pd.to_datetime(df["forecast_kst_dtm"]).dt.month
    df["high_variance_season"] = month.isin([1, 2, 3, 4, 5, 7]).astype(int)
    df["ws_x_season"] = df["ldaps_ws_10m"] * df["high_variance_season"]
    df["disagreement_x_season"] = df["ws_disagreement"] * df["high_variance_season"]
    return df

def build_datasets():
    train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
    sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")
    ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
    gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
    ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
    gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")
    train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
    sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

    raw = pd.read_excel(DATA_DIR / "info.xlsx", sheet_name="info", header=None)
    info = raw.iloc[4:21].copy()
    info.columns = raw.iloc[3]
    info = info.reset_index(drop=True)
    info = info.rename(columns={"좌표(Google)": "coord", "KPX그룹": "kpx_group", "설비용량(MW)": "capacity_mw"})
    info["kpx_group"] = info["kpx_group"].ffill().astype(int)
    coords = info["coord"].apply(dms_to_decimal)
    info["lat"] = coords.apply(lambda x: x[0])
    info["lon"] = coords.apply(lambda x: x[1])
    assert info.groupby("kpx_group")["capacity_mw"].sum().tolist() == [21.6, 21.6, 21.0]

    group_coords = info.groupby("kpx_group")[["lat", "lon"]].mean().reset_index()
    group_coords = group_coords.rename(columns={"kpx_group": "KPX그룹"})
    ldaps_group_grid = find_nearest_grid(ldaps_train, group_coords, "KPX그룹")
    gfs_group_grid = find_nearest_grid(gfs_train, group_coords, "KPX그룹")

    LDAPS_UV_PAIRS = [("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m")]
    GFS_UV_PAIRS = [
        ("heightAboveGround_10_10u", "heightAboveGround_10_10v", "10m"),
        ("heightAboveGround_80_u", "heightAboveGround_80_v", "80m"),
        ("heightAboveGround_100_100u", "heightAboveGround_100_100v", "100m"),
        ("planetaryBoundaryLayer_0_u", "planetaryBoundaryLayer_0_v", "pbl"),
    ]
    ldaps_train = add_wind_features(ldaps_train, LDAPS_UV_PAIRS)
    ldaps_test = add_wind_features(ldaps_test, LDAPS_UV_PAIRS)
    gfs_train = add_wind_features(gfs_train, GFS_UV_PAIRS)
    gfs_test = add_wind_features(gfs_test, GFS_UV_PAIRS)
    for df in [gfs_train, gfs_test]:
        df["wind_shear_100_10"] = df["ws_100m"] - df["ws_10m"]
        df["wind_shear_80_10"] = df["ws_80m"] - df["ws_10m"]

    ldaps_group_train = build_group_weather(ldaps_train, ldaps_group_grid, "ldaps")
    ldaps_group_test = build_group_weather(ldaps_test, ldaps_group_grid, "ldaps")
    gfs_group_train = build_group_weather(gfs_train, gfs_group_grid, "gfs")
    gfs_group_test = build_group_weather(gfs_test, gfs_group_grid, "gfs")

    train_labels_long = train_labels.melt(id_vars=["kst_dtm"], value_vars=TARGET_COLS,
                                          var_name="kpx_group_col", value_name="target")
    train_labels_long["kpx_group"] = train_labels_long["kpx_group_col"].str.extract(r"(\d+)").astype(int)
    train_labels_long = train_labels_long.rename(columns={"kst_dtm": "forecast_kst_dtm"}).drop(columns=["kpx_group_col"])

    test_base = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
    test_long = pd.concat([test_base.assign(kpx_group=g) for g in [1, 2, 3]], ignore_index=True)

    train_df = train_labels_long.merge(ldaps_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left") \
                                  .merge(gfs_group_train, on=["forecast_kst_dtm", "kpx_group"], how="left")
    test_df = test_long.merge(ldaps_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left") \
                        .merge(gfs_group_test, on=["forecast_kst_dtm", "kpx_group"], how="left")
    assert train_df.shape[0] == train_labels_long.shape[0]

    train_df = add_seasonal_interaction(add_forecast_disagreement(train_df))
    test_df = add_seasonal_interaction(add_forecast_disagreement(test_df))
    weather_cols = [c for c in train_df.columns if c not in ["forecast_kst_dtm", "kpx_group", "target"]]
    print(f"v10 피처 개수: {len(weather_cols)}")
    return train_df, test_df, weather_cols, sample_submission, info

### 3. 평가 지표·편향 보정·검증 fold

In [ ]:
# 2. 공통 유틸 — metric, 편향보정, fold
def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"], out["day"], out["hour"] = dt.dt.month, dt.dt.day, dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2*np.pi*out["hour"]/24); out["hour_cos"] = np.cos(2*np.pi*out["hour"]/24)
    out["month_sin"] = np.sin(2*np.pi*out["month"]/12); out["month_cos"] = np.cos(2*np.pi*out["month"]/12)
    return out

def metric(answer_df, pred_df):
    group_nmae, group_ficr = [], []
    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float); forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]; valid = actual >= capacity * 0.10
        actual, forecast = actual[valid], forecast[valid]
        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))
        unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
        group_ficr.append(np.sum(actual * unit_price) / np.sum(actual * 4.0))
    one_minus_nmae = 1 - np.mean(group_nmae); ficr = np.mean(group_ficr)
    return 0.5 * one_minus_nmae + 0.5 * ficr, one_minus_nmae, ficr

def make_sample_weight(y, capacity, low_weight=0.05):
    return np.where(y >= capacity * 0.10, 1.0, low_weight)

def apply_rolling_smoothing(pred_series, window=3, alpha=0.5):
    rolling = pred_series.rolling(window, center=True, min_periods=1).mean()
    return alpha * pred_series + (1 - alpha) * rolling

def apply_bias_correction_v2(pred, ws, group, bias_map):
    pred = pred.copy()
    if group not in bias_map: return pred
    for interval, bias in bias_map[group].items():
        mask = (ws > interval.left) & (ws <= interval.right)
        pred[mask] = pred[mask] * (1 - bias)
    return pred

def lgbm_params(n_estimators, n_jobs, seed, objective="mae"):
    # GPU(OpenCL) 벤치마크: CPU 143s -> GPU 74s (60k행x352열, 500트리 기준)
    return dict(n_estimators=n_estimators, max_depth=6, learning_rate=0.03, num_leaves=31,
                min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                random_state=seed, n_jobs=n_jobs, verbose=-1, objective=objective, device="gpu")

def prepare_matrices(tr_g, va_g, weather_cols):
    tr_feat = pd.concat([calendar_features(tr_g["forecast_kst_dtm"]).reset_index(drop=True),
                         tr_g[weather_cols].reset_index(drop=True)], axis=1)
    va_feat = pd.concat([calendar_features(va_g["forecast_kst_dtm"]).reset_index(drop=True),
                         va_g[weather_cols].reset_index(drop=True)], axis=1)
    imputer = SimpleImputer(strategy="median")
    X_tr = pd.DataFrame(imputer.fit_transform(tr_feat), columns=tr_feat.columns)
    X_va = pd.DataFrame(imputer.transform(va_feat), columns=va_feat.columns)
    return X_tr, X_va

FOLDS_Q = [
    {"name": "fold_2024Q1", "train_end": pd.Timestamp("2024-01-01"), "valid_start": pd.Timestamp("2024-01-01 01:00:00"), "valid_end": pd.Timestamp("2024-04-01")},
    {"name": "fold_2024Q2", "train_end": pd.Timestamp("2024-04-01"), "valid_start": pd.Timestamp("2024-04-01 01:00:00"), "valid_end": pd.Timestamp("2024-07-01")},
    {"name": "fold_2024Q3", "train_end": pd.Timestamp("2024-07-01"), "valid_start": pd.Timestamp("2024-07-01 01:00:00"), "valid_end": pd.Timestamp("2024-10-01")},
    {"name": "fold_2024Q4", "train_end": pd.Timestamp("2024-10-01"), "valid_start": pd.Timestamp("2024-10-01 01:00:00"), "valid_end": pd.Timestamp("2025-01-01")},
]
HOLD_FOLD = {"name": "hold_2024full", "train_end": pd.Timestamp("2024-01-01"),
             "valid_start": pd.Timestamp("2024-01-01 01:00:00"), "valid_end": pd.Timestamp("2025-01-01")}
FOLD_LIST = [(f["name"], f) for f in FOLDS_Q] + [(HOLD_FOLD["name"], HOLD_FOLD)]

COARSE_BINS = [0, 11, 13, 100]

def estimate_bias_data(train_df, weather_cols, tr_end, n_estimators, n_jobs,
                       low_weight=0.05, holdout_ratio=0.25, min_fit_days=90):
    out = {}
    for group in [1, 2, 3]:
        capacity = CAPACITY_KWH[f"kpx_group_{group}"]
        g_data = train_df[(train_df["kpx_group"] == group) & (train_df["forecast_kst_dtm"] <= tr_end)]
        g_data = g_data[g_data["target"].notna()]
        if len(g_data) == 0: continue
        first_date = g_data["forecast_kst_dtm"].min()
        available_days = max((tr_end - first_date).days, 1)
        holdout_days = min(int(available_days * holdout_ratio), 180)
        if available_days - holdout_days < min_fit_days:
            holdout_days = max(0, available_days - min_fit_days)
        tr_sub_end = tr_end - pd.Timedelta(days=holdout_days)
        fit_g = g_data[g_data["forecast_kst_dtm"] <= tr_sub_end]
        check_g = g_data[g_data["forecast_kst_dtm"] > tr_sub_end]
        if len(fit_g) == 0 or len(check_g) == 0: continue
        fit_feat = pd.concat([calendar_features(fit_g["forecast_kst_dtm"]).reset_index(drop=True),
                              fit_g[weather_cols].reset_index(drop=True)], axis=1)
        check_feat = pd.concat([calendar_features(check_g["forecast_kst_dtm"]).reset_index(drop=True),
                                check_g[weather_cols].reset_index(drop=True)], axis=1)
        imputer = SimpleImputer(strategy="median")
        X_fit = pd.DataFrame(imputer.fit_transform(fit_feat), columns=fit_feat.columns)
        X_check = pd.DataFrame(imputer.transform(check_feat), columns=check_feat.columns)
        w_fit = make_sample_weight(fit_g["target"].to_numpy(), capacity, low_weight)
        model = lgb.LGBMRegressor(**lgbm_params(n_estimators, n_jobs, SEED))
        model.fit(X_fit, fit_g["target"], sample_weight=w_fit)
        pred_check = model.predict(X_check)
        check_g = check_g.copy()
        check_g["signed_error_rate"] = (pred_check - check_g["target"]) / capacity
        check_g = check_g.loc[check_g["target"] >= capacity * 0.10]
        if len(check_g) == 0: continue
        out[group] = check_g[["ldaps_ws_10m", "signed_error_rate"]]
    return out

def bias_map_from_data(bias_data, bins_by_group):
    bias_map = {}
    for group, df in bias_data.items():
        ws_bin = pd.cut(df["ldaps_ws_10m"], bins=bins_by_group[group])
        bias_map[group] = df.groupby(ws_bin, observed=True)["signed_error_rate"].mean().to_dict()
    return bias_map

### 4. 물리·시계열·공간 피처

In [ ]:
# 3. v10TPS 피처 (T+P+S 블록)
def add_physics(df):
    df = df.copy()
    rho = df["ldaps_surface_0_sp"] / (287.05 * df["ldaps_heightAboveGround_2_t"])
    df["phy_density"] = rho
    df["phy_ws10_cu"] = df["ldaps_ws_10m"] ** 3
    df["phy_ws100_cu"] = df["gfs_ws_100m"] ** 3
    df["phy_rho_ws100_cu"] = rho * df["phy_ws100_cu"]
    df["phy_pc100"] = np.clip(df["gfs_ws_100m"], 0, 12) ** 3 / 12 ** 3
    ws50max = np.hypot(df["ldaps_heightAboveGround_50_50MUmax"], df["ldaps_heightAboveGround_50_50MVmax"])
    ws50min = np.hypot(df["ldaps_heightAboveGround_50_50MUmin"], df["ldaps_heightAboveGround_50_50MVmin"])
    df["phy_ws50max"] = ws50max
    df["phy_gust_range"] = ws50max - ws50min
    df["phy_gust_over_ws"] = ws50max / (df["ldaps_ws_10m"] + 0.5)
    df["phy_gfs_gust_factor"] = df["gfs_surface_0_gust"] / (df["gfs_ws_10m"] + 0.5)
    ws850 = np.hypot(df["gfs_isobaricInhPa_850_u"], df["gfs_isobaricInhPa_850_v"])
    df["phy_ws850"] = ws850
    df["phy_shear_850_100"] = ws850 - df["gfs_ws_100m"]
    cols = [c for c in df.columns if c.startswith("phy_")]
    return df, cols

def add_temporal(df):
    df = df.sort_values(["kpx_group", "forecast_kst_dtm"]).copy()
    for src in ["ldaps_ws_10m", "gfs_ws_100m"]:
        g = df.groupby("kpx_group")[src]
        p = "tmp_" + ("l10" if src.startswith("ldaps") else "g100")
        df[f"{p}_r3_mean"] = g.transform(lambda s: s.rolling(3, center=True, min_periods=1).mean())
        df[f"{p}_r3_std"] = g.transform(lambda s: s.rolling(3, center=True, min_periods=1).std())
        df[f"{p}_r7_mean"] = g.transform(lambda s: s.rolling(7, center=True, min_periods=1).mean())
        df[f"{p}_r7_std"] = g.transform(lambda s: s.rolling(7, center=True, min_periods=1).std())
        df[f"{p}_d1"] = g.transform(lambda s: s.diff(1))
        df[f"{p}_dm1"] = g.transform(lambda s: s.diff(-1))
        df[f"{p}_d3"] = g.transform(lambda s: s.diff(3))
        df[f"{p}_dm3"] = g.transform(lambda s: s.diff(-3))
    day_key = (pd.to_datetime(df["forecast_kst_dtm"]) - pd.Timedelta(hours=1)).dt.date
    day_grp = df.groupby([df["kpx_group"], day_key])["ldaps_ws_10m"]
    df["tmp_day_mean"] = day_grp.transform("mean")
    df["tmp_day_max"] = day_grp.transform("max")
    df["tmp_day_std"] = day_grp.transform("std")
    cols = [c for c in df.columns if c.startswith("tmp_")]
    return df, cols

def build_spatial_frames(info):
    centroids = info.groupby("kpx_group")[["lat", "lon"]].mean()
    out = {}
    for split, ld_path, gf_path in [("train", TRAIN_DIR / "ldaps_train.csv", TRAIN_DIR / "gfs_train.csv"),
                                    ("test", TEST_DIR / "ldaps_test.csv", TEST_DIR / "gfs_test.csv")]:
        ld = pd.read_csv(ld_path, encoding="utf-8-sig",
                         usecols=["forecast_kst_dtm", "grid_id", "latitude", "longitude",
                                  "heightAboveGround_10_10u", "heightAboveGround_10_10v"])
        ld["forecast_kst_dtm"] = pd.to_datetime(ld["forecast_kst_dtm"])
        ld["ws"] = np.hypot(ld["heightAboveGround_10_10u"], ld["heightAboveGround_10_10v"])
        grids = ld[["grid_id", "latitude", "longitude"]].drop_duplicates().set_index("grid_id")
        frames = []
        for group in [1, 2, 3]:
            d = haversine(centroids.loc[group, "lat"], centroids.loc[group, "lon"], grids["latitude"], grids["longitude"])
            near = d.nsmallest(4).index
            sub = ld[ld["grid_id"].isin(near)].groupby("forecast_kst_dtm")["ws"]
            agg = pd.DataFrame({"spa_l10_mean": sub.mean(), "spa_l10_std": sub.std(),
                                "spa_l10_range": sub.max() - sub.min()}).reset_index()
            agg["kpx_group"] = group
            frames.append(agg)
        ldaps_spa = pd.concat(frames, ignore_index=True)
        gf = pd.read_csv(gf_path, encoding="utf-8-sig",
                         usecols=["forecast_kst_dtm", "grid_id", "heightAboveGround_100_100u", "heightAboveGround_100_100v"])
        gf["forecast_kst_dtm"] = pd.to_datetime(gf["forecast_kst_dtm"])
        gf["ws100"] = np.hypot(gf["heightAboveGround_100_100u"], gf["heightAboveGround_100_100v"])
        sub = gf.groupby("forecast_kst_dtm")["ws100"]
        gfs_spa = pd.DataFrame({"spa_g100_mean": sub.mean(), "spa_g100_std": sub.std()}).reset_index()
        out[split] = (ldaps_spa, gfs_spa)
    return out

def build_v11(info):
    train_df, test_df, weather_cols_v10, sample_submission, _ = build_datasets()
    spa = build_spatial_frames(info)
    dfs = {}
    for split, df in [("train", train_df), ("test", test_df)]:
        df, phy_cols = add_physics(df)
        df, tmp_cols = add_temporal(df)
        ldaps_spa, gfs_spa = spa[split]
        df = df.merge(ldaps_spa, on=["forecast_kst_dtm", "kpx_group"], how="left")
        df = df.merge(gfs_spa, on="forecast_kst_dtm", how="left")
        dfs[split] = df
    spa_cols = ["spa_l10_mean", "spa_l10_std", "spa_l10_range", "spa_g100_mean", "spa_g100_std"]
    v10tps_cols = weather_cols_v10 + tmp_cols + phy_cols + spa_cols
    return dfs["train"], dfs["test"], v10tps_cols, sample_submission

### 5. 다격자 편차 피처

In [ ]:
# 4. spd 다격자 편차 피처
SPD_PATH = BASE_CACHE_DIR / "spatial_spd.parquet"
LD_VARS = {
    "ws10": lambda d: np.hypot(d["heightAboveGround_10_10u"], d["heightAboveGround_10_10v"]),
    "u10": lambda d: d["heightAboveGround_10_10u"], "v10": lambda d: d["heightAboveGround_10_10v"],
    "ws50x": lambda d: np.hypot(d["heightAboveGround_50_50MUmax"], d["heightAboveGround_50_50MVmax"]),
    "wsbl": lambda d: np.hypot(d["heightAboveGround_5_XBLWS"], d["heightAboveGround_5_YBLWS"]),
    "t2": lambda d: d["heightAboveGround_2_t"], "sp": lambda d: d["surface_0_sp"], "blh": lambda d: d["etc_0_blh"],
}
LD_SRC = ["heightAboveGround_10_10u", "heightAboveGround_10_10v", "heightAboveGround_50_50MUmax",
          "heightAboveGround_50_50MVmax", "heightAboveGround_5_XBLWS", "heightAboveGround_5_YBLWS",
          "heightAboveGround_2_t", "surface_0_sp", "etc_0_blh"]
GF_VARS = {
    "ws10": lambda d: np.hypot(d["heightAboveGround_10_10u"], d["heightAboveGround_10_10v"]),
    "ws100": lambda d: np.hypot(d["heightAboveGround_100_100u"], d["heightAboveGround_100_100v"]),
    "u100": lambda d: d["heightAboveGround_100_100u"], "v100": lambda d: d["heightAboveGround_100_100v"],
    "gust": lambda d: d["surface_0_gust"],
    "ws850": lambda d: np.hypot(d["isobaricInhPa_850_u"], d["isobaricInhPa_850_v"]),
    "t850": lambda d: d["isobaricInhPa_850_t"], "prmsl": lambda d: d["meanSea_0_prmsl"],
}
GF_SRC = ["heightAboveGround_10_10u", "heightAboveGround_10_10v", "heightAboveGround_100_100u",
          "heightAboveGround_100_100v", "surface_0_gust", "isobaricInhPa_850_u",
          "isobaricInhPa_850_v", "isobaricInhPa_850_t", "meanSea_0_prmsl"]

def _load(paths, src_cols):
    frames = []
    for p in paths:
        d = pd.read_csv(p, encoding="utf-8-sig", usecols=["forecast_kst_dtm", "grid_id", "latitude", "longitude", *src_cols])
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    return df

def _pivot(df, fn, master_idx, gid):
    v = df[["forecast_kst_dtm", "grid_id"]].copy()
    v["val"] = np.asarray(fn(df), dtype=np.float64)
    M = v.pivot_table(index="forecast_kst_dtm", columns="grid_id", values="val", aggfunc="mean")
    M = M.reindex(index=master_idx, columns=gid)
    row_mean = M.mean(axis=1)
    return M.apply(lambda c: c.fillna(row_mean))

def build_spatial(prefix, paths, src_cols, var_defs):
    df = _load(paths, src_cols)
    grids = df[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id")
    gid = grids["grid_id"].to_numpy()
    master_idx = pd.Index(np.sort(df["forecast_kst_dtm"].unique()), name="forecast_kst_dtm")
    spd_cols = {}
    for name, fn in var_defs.items():
        M = _pivot(df, fn, master_idx, gid)
        V = M.to_numpy()
        mean = V.mean(axis=1)
        spd_cols[f"sd{prefix}_{name}_mean"] = mean
        for j, g in enumerate(gid):
            spd_cols[f"sd{prefix}_{name}_g{int(g)}"] = V[:, j] - mean
    return pd.DataFrame(spd_cols, index=master_idx).reset_index()

def build_spd(force=False):
    if not force and SPD_PATH.exists():
        spd = pd.read_parquet(SPD_PATH)
    else:
        t0 = time.time()
        ld_spd = build_spatial("L", [TRAIN_DIR / "ldaps_train.csv", TEST_DIR / "ldaps_test.csv"], LD_SRC, LD_VARS)
        gf_spd = build_spatial("G", [TRAIN_DIR / "gfs_train.csv", TEST_DIR / "gfs_test.csv"], GF_SRC, GF_VARS)
        spd = ld_spd.merge(gf_spd, on="forecast_kst_dtm", how="outer")
        spd.to_parquet(SPD_PATH, index=False)
        print(f"spd 캐시 생성 완료 ({time.time()-t0:.0f}s)")
    spd_cols = [c for c in spd.columns if c != "forecast_kst_dtm"]
    return spd, spd_cols

### 6. JMA 외부 예보

In [ ]:
# 5. JMA 외부예보 수집
import requests

JMA_PATH = EXTERNAL_DIR
JMA_PATH.mkdir(parents=True, exist_ok=True)
BASE_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"
MODELS = {"jma_gsm": ["2022", "2023", "2024", "2025"], "jma_msm": ["2023", "2024", "2025"]}
HOURLY_VARS = ["wind_speed_10m", "wind_direction_10m", "temperature_2m", "surface_pressure"]
LEADS = ["previous_day2"]

def fetch_json(url, retries=3):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=120)
            d = r.json()
            if "hourly" in d:
                return d
            if "reason" in d:
                raise RuntimeError(f"API error: {d['reason']}")
        except Exception as e:
            print(f"  재시도 {attempt+1}: {e}")
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"fetch 실패: {url[:120]}")

def get_centroids(info):
    return info.groupby("kpx_group")[["lat", "lon"]].mean()

def fetch_jma_prevruns(info, force=False):
    force = force or FORCE_JMA_FETCH
    out_path = JMA_PATH / "jma_prevruns.csv"
    if not ACK_OPEN_METEO_TERMS:
        raise RuntimeError(
            "Open-Meteo 이용 조건을 팀에서 확인한 뒤 1번 셀의 "
            "ACK_OPEN_METEO_TERMS=True로 바꾸세요."
        )
    if not force and out_path.exists():
        print(f"JMA 캐시 로드: {out_path}")
        return pd.read_csv(out_path, encoding="utf-8-sig", parse_dates=["forecast_kst_dtm"])
    centroids = get_centroids(info)
    hourly_param = ",".join(f"{v}_{lead}" for v in HOURLY_VARS for lead in LEADS)
    frames = []
    for group in [1, 2, 3]:
        lat = round(float(centroids.loc[group, "lat"]), 4)
        lon = round(float(centroids.loc[group, "lon"]), 4)
        for model, years in MODELS.items():
            for year in years:
                url = (f"{BASE_URL}?latitude={lat}&longitude={lon}&models={model}"
                       f"&hourly={hourly_param}&wind_speed_unit=ms"
                       f"&start_date={year}-01-01&end_date={year}-12-31&timezone=Asia%2FSeoul")
                d = fetch_json(url)
                h = d["hourly"]
                df = pd.DataFrame({"forecast_kst_dtm": pd.to_datetime(h["time"])})
                for v in HOURLY_VARS:
                    for lead in LEADS:
                        short = {"previous_day2": "d2", "previous_day1": "d1"}[lead]
                        df[f"{v}_{short}"] = h.get(f"{v}_{lead}")
                df["kpx_group"] = group
                df["model"] = model
                frames.append(df)
                time.sleep(1)
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(out_path, index=False, encoding="utf-8-sig")
    jma_meta = {
        "source": "Open-Meteo Previous Runs API",
        "source_url": "https://open-meteo.com/en/docs/previous-runs-api",
        "terms_url": "https://open-meteo.com/en/terms",
        "lead": "previous_day2",
        "models": MODELS,
        "variables": HOURLY_VARS,
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (JMA_PATH / "jma_fetch_meta.json").write_text(
        json.dumps(jma_meta, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Saved: {out_path} ({len(out)}행)")
    return out

def build_jma_frame(jma_df):
    frames = []
    for model, tag in [("jma_gsm", "jgsm"), ("jma_msm", "jmsm")]:
        sub = jma_df[jma_df["model"] == model].copy()
        wd_rad = np.deg2rad(sub["wind_direction_10m_d2"])
        out = pd.DataFrame({
            "forecast_kst_dtm": sub["forecast_kst_dtm"], "kpx_group": sub["kpx_group"],
            f"{tag}_ws": sub["wind_speed_10m_d2"], f"{tag}_wdsin": np.sin(wd_rad), f"{tag}_wdcos": np.cos(wd_rad),
        })
        if model == "jma_gsm":
            out[f"{tag}_t2m"] = sub["temperature_2m_d2"]
            out[f"{tag}_sp"] = sub["surface_pressure_d2"]
        frames.append(out)
    return frames[0].merge(frames[1], on=["forecast_kst_dtm", "kpx_group"], how="outer")

JG_COLS = ["jgsm_ws", "jgsm_wdsin", "jgsm_wdcos", "jgsm_t2m", "jgsm_sp", "jgsm_dis"]
JM_COLS = ["jmsm_ws", "jmsm_wdsin", "jmsm_wdcos", "jmsm_dis"]

### 7. ECMWF HRES 외부 예보

In [ ]:
# 5. ECMWF IFS HRES 외부예보 수집 (open-data S3 아카이브, D-2 12z oper 런)
#   wonwoo 브랜치 exp054_fetch_ecmwf.py / exp054_gate.py 이식.
#   - 규정: D-2일 12z 런 사용 → 예측기준시점(전날 14시 KST) 이전 발행 완료.
#   - Windows에서는 eccodes DLL이 앱 제어 정책에 막혀 fetch/decode를 분리했었지만,
#     이 환경(Linux)에서는 eccodes가 pip만으로 바로 동작해 한 번에 처리한다.
#   - GBM 전용 피처 4개: ec_ws, ec_wdsin, ec_wdcos, ec_dis (DL은 안 받음 — exp-040 교훈:
#     같은 정보를 GBM·DL 양쪽에 주면 오히려 탈상관이 죽어 상쇄된다).
import json
import eccodes as _eccodes
from concurrent.futures import ThreadPoolExecutor, as_completed

EC_PATH = EXTERNAL_DIR
EC_PATH.mkdir(parents=True, exist_ok=True)
EC_RAW_PARQUET = EC_PATH / "ecmwf_prevruns.parquet"
EC_S3 = "https://ecmwf-forecasts.s3.eu-central-1.amazonaws.com"
EC_STEPS = list(range(27, 52, 3))          # 27~51h, 3시간 간격 9개
EC_START, EC_END = "2023-01-18", "2025-12-30"   # 커버리지 시작(아카이브 실측) ~ 테스트기간 D-2 끝

_ec_sess = requests.Session()
_ec_sess.headers["User-Agent"] = "baram2026-simplified"

def _ec_http_get(url, rng=None, retries=6):
    # S3 아카이브가 간헐적으로 503(SlowDown/일시 과부하)을 반환함 — 소규모 스모크 테스트에서
    # 동일 URL이 재시도 시 바로 200으로 성공하는 걸 확인, 재시도 강화로 대응 (지수 백오프, 최대 20초).
    for a in range(retries):
        try:
            h = {"Range": f"bytes={rng[0]}-{rng[0]+rng[1]-1}"} if rng else {}
            r = _ec_sess.get(url, headers=h, timeout=60)
            if r.status_code in (200, 206):
                return r.content
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep(min(2 * (2 ** a), 20))
    return None

def _ec_bilinear(msg_bytes, pts):
    """GRIB 메시지에서 각 (lat, lon) 지점의 이중선형 보간값"""
    gid = _eccodes.codes_new_from_message(msg_bytes)
    try:
        ni = _eccodes.codes_get(gid, "Ni"); nj = _eccodes.codes_get(gid, "Nj")
        lat0 = _eccodes.codes_get(gid, "latitudeOfFirstGridPointInDegrees")
        lon0 = _eccodes.codes_get(gid, "longitudeOfFirstGridPointInDegrees")
        dlat = _eccodes.codes_get(gid, "jDirectionIncrementInDegrees")
        dlon = _eccodes.codes_get(gid, "iDirectionIncrementInDegrees")
        jpos = _eccodes.codes_get(gid, "jScansPositively")
        vals = _eccodes.codes_get_values(gid).reshape(nj, ni)
    finally:
        _eccodes.codes_release(gid)
    slat = dlat if jpos else -dlat
    out = []
    for lat, lon in pts:
        fi = ((lon - lon0) % 360.0) / dlon
        fj = (lat - lat0) / slat
        i0, j0 = int(np.floor(fi)), int(np.floor(fj))
        wi, wj = fi - i0, fj - j0
        j0 = min(max(j0, 0), nj - 2)
        v = ((1 - wi) * (1 - wj) * vals[j0, i0 % ni] + wi * (1 - wj) * vals[j0, (i0 + 1) % ni]
             + (1 - wi) * wj * vals[j0 + 1, i0 % ni] + wi * wj * vals[j0 + 1, (i0 + 1) % ni])
        out.append(float(v))
    return out

def _ec_fetch_day(init_d, pts):
    d = init_d.strftime("%Y%m%d")
    roots = (["12z/0p4-beta/oper", "12z/ifs/0p4-beta/oper", "12z/ifs/0p25/oper"]
             if init_d < pd.Timestamp("2024-02-01") else
             ["12z/ifs/0p25/oper", "12z/ifs/0p4-beta/oper", "12z/0p4-beta/oper"])
    root, idx0 = None, None
    for cand in roots:
        idx0 = _ec_http_get(f"{EC_S3}/{d}/{cand}/{d}120000-27h-oper-fc.index", retries=5)
        if idx0:
            root = cand; break
    if root is None:
        return []
    rows = []
    for s in EC_STEPS:
        raw = idx0 if s == 27 else _ec_http_get(f"{EC_S3}/{d}/{root}/{d}120000-{s}h-oper-fc.index")
        if raw is None:
            continue
        ranges = {}
        for line in raw.decode("utf-8", "ignore").splitlines():
            try:
                e = json.loads(line)
            except json.JSONDecodeError:
                continue
            if e.get("levtype") == "sfc" and e.get("param") in ("10u", "10v"):
                ranges[e["param"]] = (int(e["_offset"]), int(e["_length"]))
        if "10u" not in ranges or "10v" not in ranges:
            continue
        gurl = f"{EC_S3}/{d}/{root}/{d}120000-{s}h-oper-fc.grib2"
        mu = _ec_http_get(gurl, rng=ranges["10u"])
        mv = _ec_http_get(gurl, rng=ranges["10v"])
        if mu is None or mv is None:
            continue
        us = _ec_bilinear(mu, pts)
        vs = _ec_bilinear(mv, pts)
        for g, u, v in zip([1, 2, 3], us, vs):
            rows.append(dict(init_date=d, step=s, group=g, u10=u, v10=v))
    return rows

def fetch_ecmwf_prevruns(info, start=EC_START, end=EC_END, workers=20, force=False):
    force = force or FORCE_ECMWF_FETCH
    # 체크포인트: 실행 환경이 중간에 죽어도(관측됨) 이어서 진행하도록 100일마다 부분 저장.
    inits = pd.date_range(start, end, freq="D")
    all_rows, done_inits = [], set()
    if not force and EC_RAW_PARQUET.exists():
        prev = pd.read_parquet(EC_RAW_PARQUET)
        done_inits = set(prev["init_date"].unique())
        all_rows = prev.to_dict("records")
        if done_inits >= set(d.strftime("%Y%m%d") for d in inits):
            print(f"ECMWF raw 캐시 로드: {len(prev)}행, {len(done_inits)}일 (완료)")
            return prev
        print(f"ECMWF 부분 캐시 로드: {len(prev)}행, {len(done_inits)}일 — 이어서 진행", flush=True)
    todo = [d for d in inits if d.strftime("%Y%m%d") not in done_inits]
    centroids = get_centroids(info)
    pts = [(float(centroids.loc[g, "lat"]), float(centroids.loc[g, "lon"])) for g in [1, 2, 3]]
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(_ec_fetch_day, d, pts): d for d in todo}
        n_done, n_ok = 0, 0
        for fut in as_completed(futs):
            r = fut.result()
            n_done += 1
            if r:
                n_ok += 1
                all_rows.extend(r)
            if n_done % 100 == 0:
                el = time.time() - t0
                print(f"  진행 {n_done}/{len(todo)}일 ({el/60:.1f}분, 성공 {n_ok}일, {el/n_done*len(todo)/60:.0f}분 예상)", flush=True)
                pd.DataFrame(all_rows).to_parquet(EC_RAW_PARQUET, index=False)
    out = pd.DataFrame(all_rows)
    out.to_parquet(EC_RAW_PARQUET, index=False)
    ec_meta = {
        "source": "ECMWF Open Data AWS archive",
        "source_url": "https://www.ecmwf.int/en/forecasts/datasets/open-data",
        "run": "D-2 12z IFS HRES",
        "variables": ["10u", "10v"],
        "date_range": [str(start), str(end)],
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (EC_PATH / "ecmwf_fetch_meta.json").write_text(
        json.dumps(ec_meta, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"ECMWF raw 수집 완료: {len(out)}행, {out['init_date'].nunique()}/{len(inits)}일 ({(time.time()-t0)/60:.1f}분)", flush=True)
    return out

def build_ec_frame(ec_raw):
    """(init_date, step, group, u10, v10) → 시간별 KST ec_ws/ec_wdsin/ec_wdcos (exp054_gate.py build_ec_frame 이식)"""
    ec_raw = ec_raw.copy()
    ec_raw["init_date"] = pd.to_datetime(ec_raw["init_date"])
    frames = []
    for (init_d, g), sub in ec_raw.groupby(["init_date", "group"]):
        sub = sub.sort_values("step")
        if len(sub) < 2:
            continue
        anchor_h = sub["step"].to_numpy(dtype=float)             # 27~51
        target_h = np.arange(28, 52, dtype=float)                # D 01:00 ~ D+1 00:00 (24개)
        u = np.interp(target_h, anchor_h, sub["u10"].to_numpy())
        v = np.interp(target_h, anchor_h, sub["v10"].to_numpy())
        base_utc = init_d + pd.Timedelta(hours=12)
        kst = base_utc + pd.to_timedelta(target_h, unit="h") + pd.Timedelta(hours=9)
        ws = np.hypot(u, v)
        wd = np.arctan2(u, v)
        frames.append(pd.DataFrame({"forecast_kst_dtm": kst, "kpx_group": g,
                                    "ec_ws": ws, "ec_wdsin": np.sin(wd), "ec_wdcos": np.cos(wd)}))
    out = pd.concat(frames, ignore_index=True)
    out = out.drop_duplicates(["forecast_kst_dtm", "kpx_group"], keep="last")
    return out

EC_COLS = ["ec_ws", "ec_wdsin", "ec_wdcos", "ec_dis"]

### 8. MTMLP 공통 구조

In [ ]:
# 6. DL — MTMLP + q60 비대칭 손실
GIDX = {1: 0, 2: 1, 3: 2}

class MTMLP(nn.Module):
    def __init__(self, n_feat, width=384, depth=3, p_drop=0.15, emb=8):
        super().__init__()
        self.emb = nn.Embedding(3, emb)
        layers, d_in = [], n_feat + emb
        for i in range(depth):
            layers += [nn.Linear(d_in, width), nn.BatchNorm1d(width), nn.SiLU(), nn.Dropout(p_drop)]
            d_in = width
        self.trunk = nn.Sequential(*layers)
        self.heads = nn.ModuleList([nn.Linear(width, 1) for _ in range(3)])

    def forward(self, x, g):
        h = self.trunk(torch.cat([x, self.emb(g)], dim=1))
        out = torch.zeros(x.shape[0], device=x.device)
        for gi in range(3):
            m = g == gi
            if m.any(): out[m] = self.heads[gi](h[m]).squeeze(1)
        return out

DELTA, TAU = 0.05, 0.60
def _huber(e):
    a = torch.abs(e)
    return torch.where(a <= DELTA, 0.5*e*e, DELTA*(a - 0.5*DELTA))

def q60_loss(pred, tgt, w):
    e = pred - tgt
    asym = torch.where(e < 0, 2.0*TAU, 2.0*(1.0-TAU))
    return (_huber(e) * asym * w).sum() / w.sum()

DL_CFG = dict(width=384, depth=3, p_drop=0.15, epochs=120, lr=1e-3, wd=1e-4)

def make_xy(df, cols, imputer, scaler, fit=False):
    sub = df.loc[df["target"].notna()]
    caps = sub["kpx_group"].map(lambda g: CAPACITY_KWH[f"kpx_group_{int(g)}"]).to_numpy()
    Xraw = sub[cols].to_numpy(dtype=np.float32)
    Xs = scaler.fit_transform(imputer.fit_transform(Xraw)) if fit else scaler.transform(imputer.transform(Xraw))
    g = sub["kpx_group"].map(GIDX).to_numpy().astype(np.int64)
    cf = (sub["target"].to_numpy() / caps).astype(np.float32)
    w = np.where(cf < 0.10, 0.05, 1.0).astype(np.float32)
    return Xs.astype(np.float32), g, cf, w

def train_model(Xtr, gtr, ytr, wtr, width, depth, p_drop, lr, wd, epochs, bs, seed, verbose=False,
                Xes=None, ges=None, yes=None, wes=None, patience=15):
    # early stopping: wonwoo 메인 브랜치 wi09_earlystop.py 설계 이식
    # (검증 손실 patience=15 모니터링 + best_state 체크포인트 복원, 최대 epochs·Cosine 스케줄 불변)
    torch.manual_seed(seed); np.random.seed(seed)
    model = MTMLP(Xtr.shape[1], width, depth, p_drop).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xt = torch.tensor(Xtr, device=DEVICE); gt = torch.tensor(gtr, device=DEVICE)
    yt = torch.tensor(ytr, device=DEVICE); wt = torch.tensor(wtr, device=DEVICE)
    n = Xt.shape[0]
    use_es = Xes is not None
    if use_es:
        Xe = torch.tensor(Xes, device=DEVICE); ge = torch.tensor(ges, device=DEVICE)
        ye = torch.tensor(yes, device=DEVICE); we = torch.tensor(wes, device=DEVICE)
        best_loss, bad, best_state = float("inf"), 0, None
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            opt.zero_grad()
            loss = q60_loss(model(Xt[idx], gt[idx]), yt[idx], wt[idx])
            loss.backward(); opt.step()
        sched.step()
        if use_es:
            model.eval()
            with torch.no_grad():
                val_loss = q60_loss(model(Xe, ge), ye, we).item()
            if val_loss < best_loss - 1e-5:
                best_loss, bad = val_loss, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if use_es and best_state is not None:
        model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict(model, X, g, bs=8192):
    model.eval(); out = []
    Xt = torch.tensor(X, device=DEVICE); gt = torch.tensor(g, device=DEVICE)
    for i in range(0, Xt.shape[0], bs):
        out.append(model(Xt[i:i+bs], gt[i:i+bs]).cpu().numpy())
    return np.concatenate(out)

def es_val_mask(df):
    # wonwoo 메인 브랜치 wi09_earlystop.py 이식: val = 학습창 "마지막 해"의 3/6/9/12월 11~20일
    # (연속 10일×4계절=40일). 폴드 경계가 다음 해 01-01 00:00 한 시간을 포함할 수 있어 -1h 보정으로 해를 정함.
    dt = df["forecast_kst_dtm"]
    last_year = int((dt - pd.Timedelta(hours=1)).dt.year.max())
    m = (dt.dt.year == last_year) & dt.dt.month.isin((3, 6, 9, 12)) & dt.dt.day.isin(range(11, 21))
    assert m.any(), f"early-stopping val 블록 0행 (last_year={last_year})"
    return m

def dl_fold_predict(tr_df, va_df, cols, seeds, cfg=None, patience=15):
    cfg = cfg or DL_CFG
    imp = SimpleImputer(strategy="median", keep_empty_features=True)
    sca = StandardScaler()
    # early stopping: 계절 블록을 학습에서 완전히 제외하고 검증으로 사용 (fold의 va_df와는 별개, 리키지 없음)
    vm = es_val_mask(tr_df)
    tr_fit_df = tr_df.loc[~vm]
    tr_es_df = tr_df.loc[vm]
    Xtr, gtr, ytr, wtr = make_xy(tr_fit_df, cols, imp, sca, fit=True)
    Xes, ges, yes, wes = make_xy(tr_es_df, cols, imp, sca, fit=False)
    Xva = sca.transform(imp.transform(va_df[cols].to_numpy(dtype=np.float32))).astype(np.float32)
    gva = va_df["kpx_group"].map(GIDX).to_numpy().astype(np.int64)
    preds = []
    for s in range(seeds):
        m = train_model(Xtr, gtr, ytr, wtr, width=cfg["width"], depth=cfg["depth"],
                        p_drop=cfg["p_drop"], lr=cfg["lr"], wd=cfg["wd"],
                        epochs=cfg["epochs"], bs=1024, seed=s, verbose=False,
                        Xes=Xes, ges=ges, yes=yes, wes=wes, patience=patience)
        preds.append(predict(m, Xva, gva))
        del m; torch.cuda.empty_cache()
    return np.mean(preds, axis=0)

### 9. no-snap 후처리

In [ ]:
# 7. 후처리 (단순화: 편향보정 + rolling + clip, 스냅 없음)
def postprocess_simple(cf, cap, ws, group, bias_map, clip_lo=0.16, alpha=0.40):
    base = apply_bias_correction_v2(cf * cap, ws, group, bias_map)
    base = apply_rolling_smoothing(pd.Series(base), 3, alpha).to_numpy()
    return np.clip(base, cap * clip_lo, cap)

def score_from_tidy(tidy, pred_col):
    ans, prd = [], []
    for fname, sub in tidy.groupby("fold", sort=False):
        idx = np.sort(sub["forecast_kst_dtm"].unique())
        out = []
        for val in ("target", pred_col):
            m = sub.pivot_table(index="forecast_kst_dtm", columns="group", values=val)
            m.columns = [f"kpx_group_{int(c)}" for c in m.columns]
            out.append(m.reindex(index=idx, columns=TARGET_COLS).fillna(0.0))
        ans.append(out[0]); prd.append(out[1])
    return metric(pd.concat(ans, ignore_index=True), pd.concat(prd, ignore_index=True))

def build_slices(cache):
    return [((fn, int(g)), sub.sort_values("forecast_kst_dtm")) for (fn, g), sub in cache.groupby(["fold", "group"], sort=False)]

### 10. SCADA 가용성 가중치

In [ ]:
# 8. SCADA 가용성
GROUP_TURBINES = {
    1: [("vestas", i) for i in range(1, 7)],
    2: [("vestas", i) for i in range(7, 13)],
    3: [("unison", i) for i in range(1, 6)],
}

def load_scada():
    v = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
    u = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
    return v.merge(u, on="kst_dtm", how="outer").sort_values("kst_dtm").reset_index(drop=True)

def compute_availability(sc):
    sc = sc.copy()
    # FIX: floor -> ceil. 팀원 검증(SCADA합 vs kpx_group_3 라벨 상관): floor=0.949, ceil=0.997.
    # 라벨이 "종료 시각" 기준으로 타임스탬프가 찍혀있다는 뜻 — floor로 묶으면 가용성이
    # 엉뚱한 시간대(한 시간 어긋난 forecast_kst_dtm)에 붙어서 가중치가 새고 있었음.
    sc["hour"] = sc["kst_dtm"].dt.ceil("h")
    rows = []
    for g, turbines in GROUP_TURBINES.items():
        p_cols = [f"{mk}_wtg{i:02d}_power_kw10m" for mk, i in turbines]
        w_cols = [f"{mk}_wtg{i:02d}_ws" for mk, i in turbines]
        grp = sc[["hour"] + p_cols + w_cols]
        med_ws = grp[w_cols].median(axis=1)
        agg = {}
        for (mk, i), pc, wc in zip(turbines, p_cols, w_cols):
            ws_eff = grp[wc].fillna(med_ws)
            windy = ws_eff >= 5.0
            # FIX: 결측(WTG02 2024-02형, 수집 장애 가능성)과 확정 정지(WTG01 2024-12형,
            # 출력 0 실측)를 구분. 기존엔 결측도 확정 정지와 동일하게 취급해 실제로는
            # 살아있었을 수도 있는 구간까지 과도하게 저가중치 처리했음.
            is_missing = grp[pc].isna()
            prod = grp[pc] > 1.0
            dead_strict = (~prod.fillna(False)) & windy & ~is_missing
            missing_windy = is_missing & windy
            slots = grp.groupby("hour")[pc].size()
            agg[f"ds_{mk}{i}"] = dead_strict.groupby(grp["hour"]).sum()
            agg[f"ms_{mk}{i}"] = missing_windy.groupby(grp["hour"]).sum()
            agg[f"n_{mk}{i}"] = slots
        A = pd.DataFrame(agg)
        n_t = len(turbines)
        down_confirmed = sum((A[f"ds_{mk}{i}"] >= np.maximum(A[f"n_{mk}{i}"] - 1, 5)).astype(int) for mk, i in turbines)
        down_missing = sum((A[f"ms_{mk}{i}"] >= np.maximum(A[f"n_{mk}{i}"] - 1, 5)).astype(int) for mk, i in turbines)
        un_s = down_confirmed + 0.5 * down_missing  # 결측은 절반 가중치만 "정지"로 반영 (불확실성 인정)
        rows.append(pd.DataFrame({"forecast_kst_dtm": A.index, "group": g, "avail_strict": 1 - un_s / n_t}))
    return pd.concat(rows, ignore_index=True)

# NOTE: UNISON WTG01/03/04/05가 2023-12부터 출력값이 100kW 단위로 이산화되는 현상을 발견해서
# (팀원 지적 + 검증: 대조군 WTG02는 불변인데 해당 터빈들만 "풍속>=5인데 출력0" 비율이
# 5.8%->14.8%로 급증) 이산화 이후 출력0을 결측과 동일하게 절반 가중치로 낮춰봤으나
# 실측 결과 CV -0.0008 / HOLD 변화없음으로 이득이 없어 되돌림 (2단계: ceil정렬+결측분리까지만 유지).

def avail_factor(av_vals, low=0.05):
    a = np.nan_to_num(av_vals.astype(float), nan=1.0)
    return np.where(a < 0.9, low, np.where(a < 0.999, 0.3, 1.0))

def make_sample_weight_scada(y, capacity, av_series, low_weight=0.05):
    base_w = make_sample_weight(y, capacity, low_weight)
    af = avail_factor(av_series.to_numpy())
    return base_w * af

### 11. Flow-aligned 공간 피처

In [ ]:
FLOW_CACHE_PATH = CACHE_DIR / f"flow_spatial_features_{FLOW_VERSION}.parquet"
FLOW_K_NEAREST = 4

FLOW_SOURCES = {
    "L": {
        "train": TRAIN_DIR / "ldaps_train.csv",
        "test": TEST_DIR / "ldaps_test.csv",
        "u": "heightAboveGround_10_10u",
        "v": "heightAboveGround_10_10v",
    },
    "G": {
        "train": TRAIN_DIR / "gfs_train.csv",
        "test": TEST_DIR / "gfs_test.csv",
        "u": "heightAboveGround_100_100u",
        "v": "heightAboveGround_100_100v",
    },
}

def _fill_by_row(values):
    values = values.astype(np.float64, copy=True)
    row_mean = np.nanmean(values, axis=1)
    missing = np.isnan(values)
    values[missing] = np.take(row_mean, np.where(missing)[0])
    return values

def _flow_source_frame(info, source_tag, spec, k_nearest=4):
    usecols = ["forecast_kst_dtm", "grid_id", "latitude", "longitude", spec["u"], spec["v"]]
    raw = pd.concat(
        [pd.read_csv(spec["train"], encoding="utf-8-sig", usecols=usecols),
         pd.read_csv(spec["test"], encoding="utf-8-sig", usecols=usecols)],
        ignore_index=True,
    )
    raw["forecast_kst_dtm"] = pd.to_datetime(raw["forecast_kst_dtm"])
    grids = raw[["grid_id", "latitude", "longitude"]].drop_duplicates().set_index("grid_id")
    centroids = info.groupby("kpx_group")[["lat", "lon"]].mean()
    frames = []

    for group in [1, 2, 3]:
        lat0 = float(centroids.loc[group, "lat"])
        lon0 = float(centroids.loc[group, "lon"])
        distance = pd.Series(
            haversine(lat0, lon0, grids["latitude"], grids["longitude"]),
            index=grids.index,
        )
        selected = distance.nsmallest(min(k_nearest, len(distance))).index.tolist()
        grid_xy = grids.loc[selected]
        x_km = (grid_xy["longitude"].to_numpy() - lon0) * 111.0 * np.cos(np.deg2rad(lat0))
        y_km = (grid_xy["latitude"].to_numpy() - lat0) * 111.0
        design = np.column_stack([x_km, y_km, np.ones(len(selected))])
        design_pinv = np.linalg.pinv(design)

        sub = raw.loc[raw["grid_id"].isin(selected)]
        u = sub.pivot_table(index="forecast_kst_dtm", columns="grid_id", values=spec["u"], aggfunc="mean")
        v = sub.pivot_table(index="forecast_kst_dtm", columns="grid_id", values=spec["v"], aggfunc="mean")
        master_index = u.index.union(v.index).sort_values()
        u = u.reindex(index=master_index, columns=selected)
        v = v.reindex(index=master_index, columns=selected)
        U = _fill_by_row(u.to_numpy())
        V = _fill_by_row(v.to_numpy())

        coef_u = U @ design_pinv.T
        coef_v = V @ design_pinv.T
        du_dx, du_dy = coef_u[:, 0], coef_u[:, 1]
        dv_dx, dv_dy = coef_v[:, 0], coef_v[:, 1]

        mean_u = np.nanmean(U, axis=1)
        mean_v = np.nanmean(V, axis=1)
        mean_ws = np.hypot(mean_u, mean_v)
        ex = mean_u / np.maximum(mean_ws, 0.2)
        ey = mean_v / np.maximum(mean_ws, 0.2)
        nx, ny = -ey, ex

        dws_dx = (mean_u * du_dx + mean_v * dv_dx) / np.maximum(mean_ws, 0.2)
        dws_dy = (mean_u * du_dy + mean_v * dv_dy) / np.maximum(mean_ws, 0.2)
        along_grad = ex * dws_dx + ey * dws_dy
        cross_grad = nx * dws_dx + ny * dws_dy
        divergence = du_dx + dv_dy
        vorticity = dv_dx - du_dy

        node_ws = np.hypot(U, V)
        unit_u = U / np.maximum(node_ws, 0.2)
        unit_v = V / np.maximum(node_ws, 0.2)
        resultant = np.hypot(np.nanmean(unit_u, axis=1), np.nanmean(unit_v, axis=1))
        direction_spread = 1.0 - np.clip(resultant, 0.0, 1.0)
        along_node = U * ex[:, None] + V * ey[:, None]
        along_range = np.nanmax(along_node, axis=1) - np.nanmin(along_node, axis=1)

        prefix = f"flow_{source_tag}"
        frame = pd.DataFrame({
            "forecast_kst_dtm": master_index,
            "kpx_group": group,
            f"{prefix}_along_grad": along_grad,
            f"{prefix}_cross_grad": cross_grad,
            f"{prefix}_divergence": divergence,
            f"{prefix}_vorticity": vorticity,
            f"{prefix}_direction_spread": direction_spread,
            f"{prefix}_along_range": along_range,
        })
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

def build_flow_features(info, force=False):
    if FLOW_CACHE_PATH.exists() and not force:
        flow = pd.read_parquet(FLOW_CACHE_PATH)
        print("flow 피처 캐시 로드:", flow.shape)
    else:
        started = time.time()
        ld_flow = _flow_source_frame(info, "L", FLOW_SOURCES["L"], FLOW_K_NEAREST)
        gf_flow = _flow_source_frame(info, "G", FLOW_SOURCES["G"], FLOW_K_NEAREST)
        flow = ld_flow.merge(gf_flow, on=["forecast_kst_dtm", "kpx_group"], how="outer")
        flow.to_parquet(FLOW_CACHE_PATH, index=False)
        print(f"flow 피처 생성 완료: {flow.shape} ({time.time()-started:.1f}s)")
    assert not flow.duplicated(["forecast_kst_dtm", "kpx_group"]).any()
    flow_cols = [c for c in flow.columns if c.startswith("flow_")]
    constant = [c for c in flow_cols if flow[c].nunique(dropna=True) <= 1]
    assert not constant, f"상수 flow 피처: {constant}"
    return flow, flow_cols

### 12. C0·Flow 공통 데이터 준비

In [ ]:
_, _, _, sample_submission, info = build_datasets()
train_df_tps, test_df_tps, cols_tps, _ = build_v11(info)

spd, spd_cols = build_spd()
train_df_tps = train_df_tps.merge(spd, on="forecast_kst_dtm", how="left")
test_df_tps = test_df_tps.merge(spd, on="forecast_kst_dtm", how="left")

jma_raw = fetch_jma_prevruns(info)
jma = build_jma_frame(jma_raw)
train_df_tps = train_df_tps.merge(jma, on=["forecast_kst_dtm", "kpx_group"], how="left")
test_df_tps = test_df_tps.merge(jma, on=["forecast_kst_dtm", "kpx_group"], how="left")
for frame in [train_df_tps, test_df_tps]:
    frame["jgsm_dis"] = (frame["jgsm_ws"] - frame["ldaps_ws_10m"]).abs()
    frame["jmsm_dis"] = (frame["jmsm_ws"] - frame["ldaps_ws_10m"]).abs()

cols_gspd = cols_tps + JG_COLS + JM_COLS + spd_cols
ec_raw = fetch_ecmwf_prevruns(info)
ec = build_ec_frame(ec_raw)
train_df_tps = train_df_tps.merge(ec, on=["forecast_kst_dtm", "kpx_group"], how="left")
test_df_tps = test_df_tps.merge(ec, on=["forecast_kst_dtm", "kpx_group"], how="left")
for frame in [train_df_tps, test_df_tps]:
    frame["ec_dis"] = (frame["ec_ws"] - frame["ldaps_ws_10m"]).abs()

cols_c0 = cols_gspd + EC_COLS

WEAK_FEATURES = [
    'disagreement_x_season',
    'gfs_wd_cos_100m',
    'gfs_wd_cos_pbl',
    'high_variance_season',
    'jgsm_dis',
    'ldaps_etc_0_VLCDC',
    'ldaps_etc_0_lcc',
    'ldaps_heightAboveGround_10_10u',
    'ldaps_surface_0_SNOM',
    'ldaps_surface_0_lsm',
    'ldaps_surface_0_sp',
    'ldaps_wd_cos_10m',
    'ldaps_ws_10m',
    'phy_gust_over_ws',
    'phy_gust_range',
    'sdG_gust_g5',
    'sdG_t850_g5',
    'sdG_v100_g6',
    'sdG_ws100_g3',
    'sdG_ws10_g6',
    'sdG_ws10_g7',
    'sdG_ws10_g8',
    'sdG_ws10_g9',
    'sdL_blh_g1',
    'sdL_blh_g10',
    'sdL_blh_g11',
    'sdL_blh_g13',
    'sdL_blh_g14',
    'sdL_blh_g15',
    'sdL_blh_g16',
    'sdL_blh_g2',
    'sdL_blh_g3',
    'sdL_blh_g4',
    'sdL_blh_g5',
    'sdL_blh_g7',
    'sdL_blh_g8',
    'sdL_blh_g9',
    'sdL_t2_g15',
    'sdL_t2_g16',
    'sdL_t2_g2',
    'sdL_u10_g10',
    'sdL_u10_g14',
    'sdL_u10_g15',
    'sdL_u10_g16',
    'sdL_u10_g2',
    'sdL_u10_g5',
    'sdL_u10_g8',
    'sdL_v10_g10',
    'sdL_v10_g11',
    'sdL_v10_g12',
    'sdL_v10_g13',
    'sdL_v10_g14',
    'sdL_v10_g16',
    'sdL_v10_g2',
    'sdL_v10_g4',
    'sdL_v10_g7',
    'sdL_v10_g9',
    'sdL_ws10_g1',
    'sdL_ws10_g15',
    'sdL_ws10_g2',
    'sdL_ws10_g5',
    'sdL_ws10_g8',
    'sdL_ws50x_g1',
    'sdL_ws50x_g14',
    'sdL_ws50x_g5',
    'sdL_wsbl_g10',
    'sdL_wsbl_g11',
    'sdL_wsbl_g13',
    'sdL_wsbl_g14',
    'sdL_wsbl_g16',
    'sdL_wsbl_g7',
    'sdL_wsbl_g8',
    'spa_l10_range',
    'spa_l10_std',
    'tmp_g100_d1',
    'tmp_g100_dm1',
    'tmp_g100_dm3',
    'tmp_g100_r3_std',
    'tmp_l10_d1',
    'tmp_l10_d3',
    'tmp_l10_dm1',
    'tmp_l10_r3_std',
    'tmp_l10_r7_std',
]

cols_gspd = [c for c in cols_gspd if c not in WEAK_FEATURES]
cols_c0 = [c for c in cols_c0 if c not in WEAK_FEATURES]

flow_frame, flow_cols = build_flow_features(info)
train_df_tps = train_df_tps.merge(flow_frame, on=["forecast_kst_dtm", "kpx_group"], how="left")
test_df_tps = test_df_tps.merge(flow_frame, on=["forecast_kst_dtm", "kpx_group"], how="left")
cols_f2 = cols_c0 + flow_cols

assert len(cols_c0) == 269, f"C0 피처 수가 예상과 다름: {len(cols_c0)}"
assert len(flow_cols) == 12, f"flow 피처 수가 예상과 다름: {len(flow_cols)}"
assert len(cols_f2) == 281
assert set(flow_cols).isdisjoint(cols_c0)

flow_summary = pd.DataFrame({
    "feature": flow_cols,
    "train_coverage": [float(train_df_tps[c].notna().mean()) for c in flow_cols],
    "test_coverage": [float(test_df_tps[c].notna().mean()) for c in flow_cols],
    "train_mean": [float(train_df_tps[c].mean()) for c in flow_cols],
    "train_std": [float(train_df_tps[c].std()) for c in flow_cols],
})
flow_summary.to_csv(RUN_ROOT / "flow_feature_summary.csv", index=False)
display(flow_summary.round(6))

sc = load_scada()
av = compute_availability(sc)
av["forecast_kst_dtm"] = pd.to_datetime(av["forecast_kst_dtm"])



availability = av.rename(columns={"group": "kpx_group"})[
    ["forecast_kst_dtm", "kpx_group", "avail_strict"]
]
train_df_tps = train_df_tps.merge(
    availability, on=["forecast_kst_dtm", "kpx_group"], how="left"
)
train_df_tps["avail_strict"] = train_df_tps["avail_strict"].fillna(1.0)
test_df_tps["avail_strict"] = 1.0

print("train:", train_df_tps.shape, "/ test:", test_df_tps.shape)
print("C0 features:", len(cols_c0), "/ C0+Flow:", len(cols_f2))

### 13. Serial residual 실험 설정

In [ ]:
SERIAL_VERSION = "v1"
SERIAL_CACHE_DIR = CACHE_DIR / f"serial_residual_{SERIAL_VERSION}"
SERIAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SERIAL_DL_SEEDS = 3
SERIAL_GBM_SEEDS = 3
SERIAL_PATIENCE = 15
PRIMARY_OPERATION = "scaled"
PRIMARY_EXPERIMENT = "R1_FLOW"
RUN_FINAL_MODEL = True

SERIAL_PASS_CV_DELTA = 0.0005
SERIAL_PASS_MIN_POSITIVE_FOLDS = 3
SERIAL_PASS_MIN_SUBMETRIC_DELTA = -0.0005

GROUP_CAPACITY = {1: 21600, 2: 21600, 3: 21000}

SERIAL_LGB_PARAMS = {
    "objective": "mae",
    "n_estimators": 633,
    "learning_rate": 0.032229933058504565,
    "num_leaves": 36,
    "min_child_samples": 141,
    "feature_fraction": 0.8262159071232439,
    "bagging_fraction": 0.9504907401583689,
    "bagging_freq": 9,
    "lambda_l1": 0.0006025002498529692,
    "lambda_l2": 0.010273205943481366,
}

WIND_BASE_CANDIDATES = [
    "ldaps_ws_10m",
    "gfs_ws_10m",
    "gfs_ws_80m",
    "gfs_ws_100m",
    "gfs_ws_pbl",
    "gfs_surface_0_gust",
    "gfs_wind_shear_100_10",
    "gfs_wind_shear_80_10",
    "phy_ws50max",
    "phy_ws850",
    "phy_pc100",
    "spa_l10_mean",
    "spa_g100_mean",
    "tmp_l10_r3_mean",
    "tmp_l10_r7_mean",
    "tmp_g100_r3_mean",
    "tmp_g100_r7_mean",
    "tmp_day_mean",
    "tmp_day_max",
    "jgsm_ws",
    "jmsm_ws",
    "ec_ws",
]
DL_WIND_COLS = [col for col in WIND_BASE_CANDIDATES if col in train_df_tps.columns]
assert len(DL_WIND_COLS) >= 15, f"풍속 DL 입력이 너무 적음: {DL_WIND_COLS}"

EXPERIMENT_COLS = {
    "R0_CONTROL": cols_c0 + ["serial_base_cf"],
    "R1_FLOW": cols_f2 + ["serial_base_cf"],
}
assert set(flow_cols).isdisjoint(cols_c0)
assert len(EXPERIMENT_COLS["R1_FLOW"]) - len(EXPERIMENT_COLS["R0_CONTROL"]) == 12

REFERENCE_OOF_PATH = Path(
    "/content/drive/MyDrive/BARAM2026/hyejin_flow_spatial/oof_predictions.parquet"
) if IN_COLAB else Path.cwd() / "runs" / "hyejin_flow_spatial" / "oof_predictions.parquet"

print("DL wind features:", len(DL_WIND_COLS))
print(DL_WIND_COLS)
print("serial cache:", SERIAL_CACHE_DIR)

### 14. Huber DL base와 시간 2분할 교차적합

In [ ]:
SERIAL_EPOCH_ROWS = []


def serial_huber_loss(prediction, target, weight):
    return (_huber(prediction - target) * weight).sum() / weight.sum()


def serial_es_mask(frame):
    labeled = frame["target"].notna()
    dt = frame["forecast_kst_dtm"]
    last_year = int((dt.loc[labeled] - pd.Timedelta(hours=1)).dt.year.max())
    mask = (
        labeled
        & (dt.dt.year == last_year)
        & dt.dt.month.isin((3, 6, 9, 12))
        & dt.dt.day.isin(range(11, 21))
    )
    if int(mask.sum()) < 240:
        times = np.sort(dt.loc[labeled].unique())
        n_times = min(24 * 40, max(24 * 7, len(times) // 10))
        cutoff = times[-n_times]
        mask = labeled & (dt >= cutoff)
    assert mask.any() and (labeled & ~mask).any()
    return mask


def serial_xy(frame, cols, imputer, scaler, fit=False):
    sub = frame.loc[frame["target"].notna()]
    raw = sub[cols].to_numpy(dtype=np.float32)
    if fit:
        values = scaler.fit_transform(imputer.fit_transform(raw))
    else:
        values = scaler.transform(imputer.transform(raw))
    groups = sub["kpx_group"].map(GIDX).to_numpy(dtype=np.int64)
    capacity = sub["kpx_group"].map(GROUP_CAPACITY).to_numpy(dtype=float)
    target_cf = (sub["target"].to_numpy(dtype=float) / capacity).astype(np.float32)
    low_weight = np.where(target_cf < 0.10, 0.05, 1.0)
    availability_weight = avail_factor(sub["avail_strict"].to_numpy())
    weight = (low_weight * availability_weight).astype(np.float32)
    return values.astype(np.float32), groups, target_cf, weight


def train_serial_dl(
    X_train, group_train, y_train, weight_train,
    X_es, group_es, y_es, weight_es, seed,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = MTMLP(
        X_train.shape[1],
        width=DL_CFG["width"],
        depth=DL_CFG["depth"],
        p_drop=DL_CFG["p_drop"],
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=DL_CFG["lr"], weight_decay=DL_CFG["wd"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=DL_CFG["epochs"]
    )

    Xt = torch.tensor(X_train, device=DEVICE)
    gt = torch.tensor(group_train, device=DEVICE)
    yt = torch.tensor(y_train, device=DEVICE)
    wt = torch.tensor(weight_train, device=DEVICE)
    Xe = torch.tensor(X_es, device=DEVICE)
    ge = torch.tensor(group_es, device=DEVICE)
    ye = torch.tensor(y_es, device=DEVICE)
    we = torch.tensor(weight_es, device=DEVICE)

    best_loss = float("inf")
    best_state = None
    best_epoch = 0
    bad_epochs = 0
    batch_size = 1024

    for epoch in range(DL_CFG["epochs"]):
        model.train()
        permutation = torch.randperm(len(Xt), device=DEVICE)
        for start in range(0, len(Xt), batch_size):
            idx = permutation[start:start + batch_size]
            optimizer.zero_grad()
            loss = serial_huber_loss(model(Xt[idx], gt[idx]), yt[idx], wt[idx])
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_loss = serial_huber_loss(model(Xe, ge), ye, we).item()
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            best_epoch = epoch + 1
            bad_epochs = 0
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= SERIAL_PATIENCE:
                break

    assert best_state is not None
    model.load_state_dict(best_state)
    return model, best_epoch, best_loss


def serial_dl_predict(train_frame, query_frame, tag):
    validation_mask = serial_es_mask(train_frame)
    fit_frame = train_frame.loc[~validation_mask]
    es_frame = train_frame.loc[validation_mask]
    imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    scaler = StandardScaler()
    X_train, group_train, y_train, weight_train = serial_xy(
        fit_frame, DL_WIND_COLS, imputer, scaler, fit=True
    )
    X_es, group_es, y_es, weight_es = serial_xy(
        es_frame, DL_WIND_COLS, imputer, scaler, fit=False
    )
    X_query = scaler.transform(
        imputer.transform(query_frame[DL_WIND_COLS].to_numpy(dtype=np.float32))
    ).astype(np.float32)
    group_query = query_frame["kpx_group"].map(GIDX).to_numpy(dtype=np.int64)

    predictions = []
    for seed in range(SERIAL_DL_SEEDS):
        model, best_epoch, best_loss = train_serial_dl(
            X_train, group_train, y_train, weight_train,
            X_es, group_es, y_es, weight_es, seed,
        )
        predictions.append(predict(model, X_query, group_query))
        SERIAL_EPOCH_ROWS.append({
            "tag": tag,
            "seed": seed,
            "best_epoch": best_epoch,
            "best_loss": best_loss,
            "n_fit": len(fit_frame),
            "n_es": len(es_frame),
            "n_query": len(query_frame),
        })
        del model
        torch.cuda.empty_cache()
    print(tag, "best epochs:", SERIAL_EPOCH_ROWS[-SERIAL_DL_SEEDS:])
    return np.mean(predictions, axis=0)


def build_fold_base_predictions(fold_name, train_all, valid_all):
    path = SERIAL_CACHE_DIR / f"serial_base_{fold_name}.parquet"
    if path.exists():
        cached = pd.read_parquet(path)
        assert set(cached["role"].unique()) == {"train", "valid"}
        print("DL base cache:", path.name, cached.shape)
        return cached

    labeled = train_all.loc[train_all["target"].notna()].copy()
    train_signature = pd.Timestamp(labeled["forecast_kst_dtm"].max()).strftime("%Y%m%d%H%M")
    train_path = SERIAL_CACHE_DIR / f"serial_base_train_{train_signature}.parquet"

    if train_path.exists():
        train_base = pd.read_parquet(train_path)
        assert len(train_base) == len(labeled)
        assert np.isfinite(train_base["serial_base_cf"]).all()
        print("DL train crossfit cache:", train_path.name, train_base.shape)
    else:
        midpoint = labeled["forecast_kst_dtm"].quantile(0.50)
        first_half = labeled["forecast_kst_dtm"] <= midpoint
        train_prediction = np.full(len(labeled), np.nan, dtype=float)

        for fit_mask, query_mask, suffix in [
            (first_half, ~first_half, "early_to_late"),
            (~first_half, first_half, "late_to_early"),
        ]:
            train_prediction[np.flatnonzero(query_mask.to_numpy())] = serial_dl_predict(
                labeled.loc[fit_mask], labeled.loc[query_mask], f"{train_signature}:{suffix}"
            )
        assert np.isfinite(train_prediction).all()
        train_base = labeled[["kpx_group", "forecast_kst_dtm", "target"]].copy()
        train_base["serial_base_cf"] = train_prediction
        train_base["role"] = "train"
        train_base.to_parquet(train_path, index=False)

    valid_prediction = serial_dl_predict(
        labeled, valid_all, f"{fold_name}:train_to_valid"
    )
    valid_base = valid_all[["kpx_group", "forecast_kst_dtm", "target"]].copy()
    valid_base["serial_base_cf"] = valid_prediction
    valid_base["role"] = "valid"
    cached = pd.concat([train_base, valid_base], ignore_index=True)
    cached.insert(0, "fold", fold_name)
    cached.to_parquet(path, index=False)
    return cached

### 15. Optuna 설정 잔차 LightGBM

In [ ]:
def serial_lgb_predict(X_train, y_train, sample_weight, X_eval, seed, label):
    params = dict(SERIAL_LGB_PARAMS)
    params.update(random_state=seed, n_jobs=2, verbose=-1, device="gpu")
    try:
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_train, sample_weight=sample_weight)
    except lgb.basic.LightGBMError as error:
        print(f"[{label}] GPU LightGBM 실패, CPU 재시도: {str(error).splitlines()[0]}")
        params.update(device="cpu", force_col_wise=True)
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_train, sample_weight=sample_weight)
    return model.predict(X_eval)


def residual_predict(train_group, eval_group, feature_cols, group, label):
    X_train, X_eval = prepare_matrices(train_group, eval_group, feature_cols)
    capacity = GROUP_CAPACITY[group]
    target_cf = train_group["target"].to_numpy(dtype=float) / capacity
    residual_target = target_cf - train_group["serial_base_cf"].to_numpy(dtype=float)
    sample_weight = make_sample_weight_scada(
        train_group["target"].to_numpy(dtype=float),
        capacity,
        train_group["avail_strict"],
    )
    predictions = [
        serial_lgb_predict(
            X_train, residual_target, sample_weight, X_eval, seed,
            f"{label}:g{group}:s{seed}",
        )
        for seed in range(SERIAL_GBM_SEEDS)
    ]
    return np.mean(predictions, axis=0)


def estimate_fixed_bias_map(train_frame, feature_cols, train_end):
    bias_data = {}
    for group in [1, 2, 3]:
        capacity = GROUP_CAPACITY[group]
        group_frame = train_frame.loc[
            (train_frame["kpx_group"] == group)
            & (train_frame["forecast_kst_dtm"] <= train_end)
            & train_frame["target"].notna()
        ].copy()
        first_date = group_frame["forecast_kst_dtm"].min()
        available_days = max((train_end - first_date).days, 1)
        holdout_days = min(int(available_days * 0.25), 180)
        if available_days - holdout_days < 90:
            holdout_days = max(0, available_days - 90)
        split_time = train_end - pd.Timedelta(days=holdout_days)
        fit_frame = group_frame.loc[group_frame["forecast_kst_dtm"] <= split_time]
        check_frame = group_frame.loc[group_frame["forecast_kst_dtm"] > split_time].copy()
        if fit_frame.empty or check_frame.empty:
            continue
        X_fit, X_check = prepare_matrices(fit_frame, check_frame, feature_cols)
        weight = make_sample_weight_scada(
            fit_frame["target"].to_numpy(dtype=float),
            capacity,
            fit_frame["avail_strict"],
        )
        prediction = serial_lgb_predict(
            X_fit, fit_frame["target"].to_numpy(dtype=float), weight,
            X_check, SEED, f"bias:g{group}",
        )
        check_frame["signed_error_rate"] = (
            prediction - check_frame["target"].to_numpy(dtype=float)
        ) / capacity
        check_frame = check_frame.loc[check_frame["target"] >= capacity * 0.10]
        if not check_frame.empty:
            bias_data[group] = check_frame[["ldaps_ws_10m", "signed_error_rate"]]
    return bias_map_from_data(
        bias_data, {group: COARSE_BINS for group in bias_data}
    )


def merge_base(frame, base_frame, role):
    keys = ["kpx_group", "forecast_kst_dtm"]
    base = base_frame.loc[base_frame["role"] == role, keys + ["serial_base_cf"]]
    merged = frame.merge(base, on=keys, how="inner", validate="one_to_one")
    assert len(merged) == len(base), f"base 병합 실패: {len(merged)} != {len(base)}"
    return merged

### 16. C0 대 C0+Flow serial ablation

In [ ]:
PREDICTION_CACHE_PATH = SERIAL_CACHE_DIR / "serial_fold_predictions.parquet"
BIAS_CACHE_PATH = SERIAL_CACHE_DIR / "serial_fold_bias.pkl"

prediction_parts = []
bias_maps = {}
if PREDICTION_CACHE_PATH.exists():
    prediction_parts = [pd.read_parquet(PREDICTION_CACHE_PATH)]
if BIAS_CACHE_PATH.exists():
    with open(BIAS_CACHE_PATH, "rb") as stream:
        bias_maps = pickle.load(stream)

completed_folds = set()
if prediction_parts:
    cached_predictions = prediction_parts[0]
    for fold_name in cached_predictions["fold"].unique():
        experiments = set(
            cached_predictions.loc[cached_predictions["fold"] == fold_name, "experiment"]
        )
        if experiments == set(EXPERIMENT_COLS):
            completed_folds.add(fold_name)

for fold_name, fold in FOLD_LIST:
    if fold_name in completed_folds:
        print("residual cache:", fold_name)
        continue
    started = time.time()
    train_all = train_df_tps.loc[
        train_df_tps["forecast_kst_dtm"] <= fold["train_end"]
    ].copy()
    valid_all = train_df_tps.loc[
        (train_df_tps["forecast_kst_dtm"] >= fold["valid_start"])
        & (train_df_tps["forecast_kst_dtm"] <= fold["valid_end"])
        & train_df_tps["target"].notna()
    ].copy()
    base_frame = build_fold_base_predictions(fold_name, train_all, valid_all)
    train_serial = merge_base(
        train_all.loc[train_all["target"].notna()], base_frame, "train"
    )
    valid_serial = merge_base(valid_all, base_frame, "valid")

    if fold_name not in bias_maps:
        if fold_name == "hold_2024full" and "fold_2024Q1" in bias_maps:
            bias_maps[fold_name] = bias_maps["fold_2024Q1"]
            print("bias map reuse: fold_2024Q1 -> hold_2024full")
        else:
            bias_maps[fold_name] = estimate_fixed_bias_map(
                train_df_tps, cols_c0, fold["train_end"]
            )

    fold_parts = []
    for experiment, feature_cols in EXPERIMENT_COLS.items():
        for group in [1, 2, 3]:
            train_group = train_serial.loc[train_serial["kpx_group"] == group].copy()
            valid_group = valid_serial.loc[
                valid_serial["kpx_group"] == group
            ].sort_values("forecast_kst_dtm").copy()
            residual_cf = residual_predict(
                train_group, valid_group, feature_cols, group,
                f"{fold_name}:{experiment}",
            )
            raw_cf = valid_group["serial_base_cf"].to_numpy(dtype=float) + residual_cf
            capacity = GROUP_CAPACITY[group]
            unscaled = postprocess_simple(
                raw_cf,
                capacity,
                valid_group["ldaps_ws_10m"].to_numpy(dtype=float),
                group,
                bias_maps[fold_name],
            )
            scaled = np.clip(unscaled * GROUP_SCALES[group], 0, capacity)
            fold_parts.append(pd.DataFrame({
                "fold": fold_name,
                "group": group,
                "forecast_kst_dtm": valid_group["forecast_kst_dtm"].to_numpy(),
                "target": valid_group["target"].to_numpy(dtype=float),
                "serial_base_cf": valid_group["serial_base_cf"].to_numpy(dtype=float),
                "residual_cf": residual_cf,
                "raw_cf": raw_cf,
                "pred_unscaled": unscaled,
                "pred_scaled": scaled,
                "experiment": experiment,
            }))
    fold_output = pd.concat(fold_parts, ignore_index=True)
    if prediction_parts and len(prediction_parts[0]):
        current = pd.concat([prediction_parts[0], fold_output], ignore_index=True)
    else:
        current = fold_output
    prediction_parts = [current]
    current.to_parquet(PREDICTION_CACHE_PATH, index=False)
    with open(BIAS_CACHE_PATH, "wb") as stream:
        pickle.dump(bias_maps, stream)
    if SERIAL_EPOCH_ROWS:
        pd.DataFrame(SERIAL_EPOCH_ROWS).to_csv(
            RUN_ROOT / "best_epochs.csv", index=False
        )
    print(f"{fold_name} complete: {(time.time()-started)/60:.1f}분")

serial_oof = prediction_parts[0]
expected = {name for name, _ in FOLD_LIST}
assert set(serial_oof["fold"].unique()) == expected
assert serial_oof[["target", "pred_unscaled", "pred_scaled"]].notna().all().all()

### 17. 지표 비교와 gate 판정

In [ ]:
def one_group_score(frame, pred_col, group):
    capacity = GROUP_CAPACITY[group]
    actual = frame["target"].to_numpy(dtype=float)
    prediction = frame[pred_col].to_numpy(dtype=float)
    valid = actual >= capacity * 0.10
    actual, prediction = actual[valid], prediction[valid]
    error_rate = np.abs(prediction - actual) / capacity
    one_nmae = 1.0 - float(error_rate.mean())
    unit_price = np.select(
        [error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0
    )
    ficr = float(np.sum(actual * unit_price) / np.sum(actual * 4.0))
    return 0.5 * (one_nmae + ficr), one_nmae, ficr


summary_rows, fold_rows, group_rows = [], [], []
for operation, pred_col in [
    ("unscaled", "pred_unscaled"),
    ("scaled", "pred_scaled"),
]:
    for experiment in EXPERIMENT_COLS:
        experiment_frame = serial_oof.loc[serial_oof["experiment"] == experiment]
        for partition, mask in [
            ("cv", experiment_frame["fold"] != "hold_2024full"),
            ("hold", experiment_frame["fold"] == "hold_2024full"),
        ]:
            partition_frame = experiment_frame.loc[mask]
            result = score_from_tidy(partition_frame, pred_col)
            summary_rows.append({
                "operation": operation,
                "experiment": experiment,
                "partition": partition,
                "Score": result[0],
                "1-NMAE": result[1],
                "FiCR": result[2],
            })
            for group in [1, 2, 3]:
                group_frame = partition_frame.loc[partition_frame["group"] == group]
                group_result = one_group_score(group_frame, pred_col, group)
                group_rows.append({
                    "operation": operation,
                    "experiment": experiment,
                    "partition": partition,
                    "group": group,
                    "Score": group_result[0],
                    "1-NMAE": group_result[1],
                    "FiCR": group_result[2],
                })
        for fold_name, fold_frame in experiment_frame.groupby("fold"):
            fold_result = score_from_tidy(fold_frame, pred_col)
            fold_rows.append({
                "operation": operation,
                "experiment": experiment,
                "fold": fold_name,
                "Score": fold_result[0],
                "1-NMAE": fold_result[1],
                "FiCR": fold_result[2],
            })

summary = pd.DataFrame(summary_rows)
fold_scores = pd.DataFrame(fold_rows)
group_scores = pd.DataFrame(group_rows)


def add_serial_deltas(table, keys):
    metrics = ["Score", "1-NMAE", "FiCR"]
    control = table.loc[table["experiment"] == "R0_CONTROL", keys + metrics].rename(
        columns={metric_name: f"control_{metric_name}" for metric_name in metrics}
    )
    result = table.merge(control, on=keys, how="left", validate="many_to_one")
    for metric_name in metrics:
        result[f"delta_{metric_name}"] = (
            result[metric_name] - result[f"control_{metric_name}"]
        )
    return result.drop(columns=[f"control_{metric_name}" for metric_name in metrics])


summary = add_serial_deltas(summary, ["operation", "partition"])
fold_scores = add_serial_deltas(fold_scores, ["operation", "fold"])
group_scores = add_serial_deltas(group_scores, ["operation", "partition", "group"])

primary = summary.loc[
    (summary["operation"] == PRIMARY_OPERATION)
    & (summary["experiment"] == PRIMARY_EXPERIMENT)
].set_index("partition")
primary_folds = fold_scores.loc[
    (fold_scores["operation"] == PRIMARY_OPERATION)
    & (fold_scores["experiment"] == PRIMARY_EXPERIMENT)
    & fold_scores["fold"].str.startswith("fold_")
]
unscaled_primary = summary.loc[
    (summary["operation"] == "unscaled")
    & (summary["experiment"] == PRIMARY_EXPERIMENT)
].set_index("partition")
positive_folds = int((primary_folds["delta_Score"] > 0).sum())

SERIAL_FLOW_GATE_PASS = bool(
    primary.loc["cv", "delta_Score"] >= SERIAL_PASS_CV_DELTA
    and positive_folds >= SERIAL_PASS_MIN_POSITIVE_FOLDS
    and primary.loc["hold", "delta_Score"] >= 0.0
    and primary.loc["cv", "delta_1-NMAE"] >= SERIAL_PASS_MIN_SUBMETRIC_DELTA
    and primary.loc["cv", "delta_FiCR"] > 0.0
    and unscaled_primary.loc["cv", "delta_Score"] >= 0.0
)

gate_decision = {
    "passed": SERIAL_FLOW_GATE_PASS,
    "primary_operation": PRIMARY_OPERATION,
    "primary_experiment": PRIMARY_EXPERIMENT,
    "cv_delta_score": float(primary.loc["cv", "delta_Score"]),
    "cv_delta_1_nmae": float(primary.loc["cv", "delta_1-NMAE"]),
    "cv_delta_ficr": float(primary.loc["cv", "delta_FiCR"]),
    "hold_delta_score": float(primary.loc["hold", "delta_Score"]),
    "unscaled_cv_delta_score": float(unscaled_primary.loc["cv", "delta_Score"]),
    "positive_cv_folds": positive_folds,
}

reference_result = {"available": REFERENCE_OOF_PATH.exists()}
if REFERENCE_OOF_PATH.exists():
    reference_oof = pd.read_parquet(REFERENCE_OOF_PATH)
    reference_f2 = reference_oof.loc[reference_oof["experiment"] == "F2_FLOW"]
    for partition, mask in [
        ("cv", reference_f2["fold"] != "hold_2024full"),
        ("hold", reference_f2["fold"] == "hold_2024full"),
    ]:
        result = score_from_tidy(reference_f2.loc[mask], "pred")
        reference_result[partition] = {
            "Score": result[0], "1-NMAE": result[1], "FiCR": result[2]
        }

run_config = {
    "experiment": "SERIAL_RESIDUAL_FLOW",
    "serial_version": SERIAL_VERSION,
    "independent_of_wonwoo_cache": True,
    "dl": {
        "inputs": DL_WIND_COLS,
        "loss": "symmetric_huber",
        "width": DL_CFG["width"],
        "depth": DL_CFG["depth"],
        "seeds": SERIAL_DL_SEEDS,
        "crossfit": "global_time_median_2way",
    },
    "residual_lgb": {**SERIAL_LGB_PARAMS, "seeds": SERIAL_GBM_SEEDS},
    "control_feature_count": len(EXPERIMENT_COLS["R0_CONTROL"]),
    "flow_feature_count": len(EXPERIMENT_COLS["R1_FLOW"]),
    "flow_features": flow_cols,
    "primary_operation": PRIMARY_OPERATION,
}

summary.to_csv(RUN_ROOT / "summary.csv", index=False)
fold_scores.to_csv(RUN_ROOT / "fold_scores.csv", index=False)
group_scores.to_csv(RUN_ROOT / "group_scores.csv", index=False)
serial_oof.to_parquet(RUN_ROOT / "oof_serial_predictions.parquet", index=False)
(RUN_ROOT / "gate_decision.json").write_text(
    json.dumps(gate_decision, ensure_ascii=False, indent=2), encoding="utf-8"
)
(RUN_ROOT / "reference_result.json").write_text(
    json.dumps(reference_result, ensure_ascii=False, indent=2), encoding="utf-8"
)
(RUN_ROOT / "run_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8"
)

display(summary.round(6))
display(primary_folds.round(6))
display(
    group_scores.loc[
        (group_scores["operation"] == PRIMARY_OPERATION)
        & (group_scores["experiment"] == PRIMARY_EXPERIMENT)
    ].round(6)
)
print("Gate:", "PASS" if SERIAL_FLOW_GATE_PASS else "REJECT")
print(json.dumps(gate_decision, ensure_ascii=False, indent=2))

### 18. 최종 학습과 gate 기반 제출 생성

In [ ]:
SUBMISSION_PATHS = []
PRIMARY_SUBMISSION_NAME = (
    "submission_serial_flow_scaled.csv"
    if SERIAL_FLOW_GATE_PASS
    else "submission_serial_control_scaled.csv"
)


def build_final_base_predictions():
    path = SERIAL_CACHE_DIR / "serial_base_final.parquet"
    if path.exists():
        cached = pd.read_parquet(path)
        assert set(cached["role"].unique()) == {"train", "test"}
        print("final DL base cache:", cached.shape)
        return cached

    labeled = train_df_tps.loc[train_df_tps["target"].notna()].copy()
    midpoint = labeled["forecast_kst_dtm"].quantile(0.50)
    first_half = labeled["forecast_kst_dtm"] <= midpoint
    train_prediction = np.full(len(labeled), np.nan, dtype=float)
    for fit_mask, query_mask, suffix in [
        (first_half, ~first_half, "early_to_late"),
        (~first_half, first_half, "late_to_early"),
    ]:
        train_prediction[np.flatnonzero(query_mask.to_numpy())] = serial_dl_predict(
            labeled.loc[fit_mask], labeled.loc[query_mask], f"final:{suffix}"
        )
    assert np.isfinite(train_prediction).all()

    test_sorted = test_df_tps.sort_values(
        ["kpx_group", "forecast_kst_dtm"]
    ).copy()
    test_prediction = serial_dl_predict(labeled, test_sorted, "final:train_to_test")

    train_base = labeled[["kpx_group", "forecast_kst_dtm", "target"]].copy()
    train_base["serial_base_cf"] = train_prediction
    train_base["role"] = "train"
    train_base["forecast_id"] = ""
    test_base = test_sorted[
        ["forecast_id", "kpx_group", "forecast_kst_dtm"]
    ].copy()
    test_base["target"] = np.nan
    test_base["serial_base_cf"] = test_prediction
    test_base["role"] = "test"
    cached = pd.concat([train_base, test_base], ignore_index=True)
    cached.to_parquet(path, index=False)
    if SERIAL_EPOCH_ROWS:
        pd.DataFrame(SERIAL_EPOCH_ROWS).to_csv(
            RUN_ROOT / "best_epochs.csv", index=False
        )
    return cached


def make_serial_submission(components, cf_col, bias_map, scaled):
    parts = []
    for group in [1, 2, 3]:
        capacity = GROUP_CAPACITY[group]
        group_frame = components.loc[
            components["kpx_group"] == group
        ].sort_values("forecast_kst_dtm")
        prediction = postprocess_simple(
            group_frame[cf_col].to_numpy(dtype=float),
            capacity,
            group_frame["ldaps_ws_10m"].to_numpy(dtype=float),
            group,
            bias_map,
        )
        if scaled:
            prediction = np.clip(prediction * GROUP_SCALES[group], 0, capacity)
        parts.append(pd.DataFrame({
            "forecast_id": group_frame["forecast_id"].to_numpy(),
            "forecast_kst_dtm": group_frame["forecast_kst_dtm"].to_numpy(),
            "kpx_group": group,
            "pred": prediction,
        }))
    long = pd.concat(parts, ignore_index=True)
    submission = long.pivot(
        index=["forecast_id", "forecast_kst_dtm"],
        columns="kpx_group",
        values="pred",
    )
    submission.columns = [f"kpx_group_{int(group)}" for group in submission.columns]
    submission = submission.reset_index()
    submission["forecast_kst_dtm"] = pd.to_datetime(
        submission["forecast_kst_dtm"]
    ).dt.strftime("%Y-%m-%d %H:%M:%S")
    return sample_submission[["forecast_id"]].merge(
        submission, on="forecast_id", how="left"
    )


if not RUN_FINAL_MODEL:
    print("RUN_FINAL_MODEL=False: CV 결과만 저장하고 최종 학습은 생략합니다.")
else:
    if not SERIAL_FLOW_GATE_PASS:
        print("Flow gate 미통과: control만 최종 학습합니다.")
    final_base = build_final_base_predictions()
    train_final = merge_base(
        train_df_tps.loc[train_df_tps["target"].notna()], final_base, "train"
    )
    test_base = final_base.loc[final_base["role"] == "test", [
        "forecast_id", "kpx_group", "forecast_kst_dtm", "serial_base_cf"
    ]]
    test_final = test_df_tps.merge(
        test_base,
        on=["forecast_id", "kpx_group", "forecast_kst_dtm"],
        how="inner",
        validate="one_to_one",
    )
    assert len(test_final) == len(test_base)

    component_path = SERIAL_CACHE_DIR / "serial_final_components.parquet"
    bias_path = SERIAL_CACHE_DIR / "serial_final_bias.pkl"
    final_experiments = ["R0_CONTROL"]
    if SERIAL_FLOW_GATE_PASS:
        final_experiments.append("R1_FLOW")
    required_cf_cols = {
        "cf_control" if experiment == "R0_CONTROL" else "cf_flow"
        for experiment in final_experiments
    }
    cache_ready = component_path.exists() and bias_path.exists()
    if cache_ready:
        final_components = pd.read_parquet(component_path)
        cache_ready = required_cf_cols.issubset(final_components.columns)
    if cache_ready:
        with open(bias_path, "rb") as stream:
            final_bias_map = pickle.load(stream)
        print("final residual cache:", final_components.shape)
    else:
        final_bias_map = estimate_fixed_bias_map(
            train_df_tps, cols_c0, train_df_tps["forecast_kst_dtm"].max()
        )
        final_components = test_final[[
            "forecast_id", "forecast_kst_dtm", "kpx_group",
            "ldaps_ws_10m", "serial_base_cf",
        ]].copy()
        for experiment in final_experiments:
            feature_cols = EXPERIMENT_COLS[experiment]
            output_col = "cf_control" if experiment == "R0_CONTROL" else "cf_flow"
            final_components[output_col] = np.nan
            for group in [1, 2, 3]:
                train_group = train_final.loc[train_final["kpx_group"] == group].copy()
                test_group = test_final.loc[
                    test_final["kpx_group"] == group
                ].sort_values("forecast_kst_dtm").copy()
                residual_cf = residual_predict(
                    train_group, test_group, feature_cols, group,
                    f"final:{experiment}",
                )
                raw_cf = test_group["serial_base_cf"].to_numpy(dtype=float) + residual_cf
                final_components.loc[test_group.index, output_col] = raw_cf
        assert final_components[list(required_cf_cols)].notna().all().all()
        final_components.to_parquet(component_path, index=False)
        with open(bias_path, "wb") as stream:
            pickle.dump(final_bias_map, stream)

    output_specs = [
        ("cf_control", False, "submission_serial_control_unscaled.csv"),
        ("cf_control", True, "submission_serial_control_scaled.csv"),
    ]
    if SERIAL_FLOW_GATE_PASS:
        output_specs.extend([
            ("cf_flow", False, "submission_serial_flow_unscaled.csv"),
            ("cf_flow", True, "submission_serial_flow_scaled.csv"),
        ])
    for cf_col, scaled, file_name in output_specs:
        submission = make_serial_submission(
            final_components, cf_col, final_bias_map, scaled
        )
        output_path = RUN_ROOT / file_name
        submission.to_csv(output_path, index=False, encoding="utf-8-sig")
        SUBMISSION_PATHS.append(output_path)
        print("Saved:", output_path)

### 19. 결과 파일 검증

In [ ]:
required_result_paths = [
    RUN_ROOT / "summary.csv",
    RUN_ROOT / "fold_scores.csv",
    RUN_ROOT / "group_scores.csv",
    RUN_ROOT / "gate_decision.json",
    RUN_ROOT / "reference_result.json",
    RUN_ROOT / "run_config.json",
    RUN_ROOT / "oof_serial_predictions.parquet",
    RUN_ROOT / "flow_feature_summary.csv",
]
assert all(path.exists() for path in required_result_paths)

for submission_path in SUBMISSION_PATHS:
    submission = pd.read_csv(submission_path, encoding="utf-8-sig")
    assert len(submission) == len(sample_submission) == 8760
    assert submission["forecast_id"].equals(sample_submission["forecast_id"])
    assert submission[TARGET_COLS].notna().all().all()
    for col in TARGET_COLS:
        assert submission[col].between(0, CAPACITY_KWH[col]).all()
    print(submission_path.name, submission.shape, "검증 통과")

print("결과 저장 경로:", RUN_ROOT)
print("Gate:", "PASS" if SERIAL_FLOW_GATE_PASS else "REJECT")
if RUN_FINAL_MODEL:
    print("primary submission candidate:", PRIMARY_SUBMISSION_NAME)

### 생성 파일

`/content/drive/MyDrive/BARAM2026/hyejin_serial_residual_flow`에 저장됩니다.

- `summary.csv`: serial control과 Flow 후보의 CV·HOLD 비교
- `fold_scores.csv`: fold별 변화
- `group_scores.csv`: 그룹별 변화
- `gate_decision.json`: 사전 정의 gate 판정
- `reference_result.json`: 이전 Flow OOF가 있으면 참고 점수 기록
- `best_epochs.csv`: Huber DL의 seed별 best epoch
- `oof_serial_predictions.parquet`: serial OOF 예측과 구성 요소
- control은 항상 unscaled/scaled submission 2개
- gate 통과 시 Flow unscaled/scaled submission 2개 추가

Flow gate 통과 시 주 후보는 `submission_serial_flow_scaled.csv`, 미통과 시 비교 후보는
`submission_serial_control_scaled.csv`입니다. 리더보드 제출 전 CV·HOLD 결과를 먼저 확인합니다.